In [ ]:
%pip install torch torchvision torchaudio
%pip install transformers accelerate
# torchcodec은 Windows에서 ffmpeg DLL 문제로 오디오 디코딩을 깨뜨리므로 제거한다.
%pip uninstall -y torchcodec

In [ ]:
import os
import glob
import shutil
import subprocess
import numpy as np

# --- ffmpeg 자동 탐색: winget 설치 경로를 찾아 PATH에 추가 (mp3 디코딩에 필요) ---
if shutil.which("ffmpeg") is None:
    _pkgs = os.path.join(os.environ.get("LOCALAPPDATA", ""),
                         "Microsoft", "WinGet", "Packages")
    _hits = glob.glob(os.path.join(_pkgs, "Gyan.FFmpeg*", "**", "ffmpeg.exe"),
                      recursive=True)
    if _hits:
        os.environ["PATH"] = os.path.dirname(_hits[0]) + os.pathsep + os.environ["PATH"]

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline


def load_audio(path, sr=16000):
    """ffmpeg로 오디오를 디코딩해 파이프라인에 넘길 배열 딕셔너리로 반환한다.
    (파일 경로 대신 배열을 넘겨 torchcodec DLL 문제를 피한다.)"""
    ffmpeg = shutil.which("ffmpeg")
    cmd = [ffmpeg, "-nostdin", "-i", path, "-f", "f32le", "-ac", "1", "-ar", str(sr), "-"]
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    audio = np.frombuffer(raw, dtype=np.float32).copy()
    return {"raw": audio, "sampling_rate": sr}


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-base"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,   # 청크별로 타임스탬프 반환
    chunk_length_s=10,        # 입력 오디오를 10초씩 나누기
    stride_length_s=2,        # 2초씩 겹치도록 청크 나누기
)

sample = "./audio/lsy_audio_2023_58s.mp3"
result = pipe(load_audio(sample), generate_kwargs={"language": "korean"})
result

In [ ]:
start_end_text = []

for chunk in result["chunks"]:
    start = chunk[ "timestamp"][0]
    end = chunk[ "timestamp"][1]
    text = chunk[ "text"]
    start_end_text.append((start, end, text))

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("lsy_audio_2023_58.csv", index=False, sep="|")
display(df)